# Pathology Hub Curriculum Tag Hardening v11

**Workstream:** Evidence/Lesson/Research RAG — curriculum mapping + tag governance.

This notebook hardens curriculum tags by using ABPath as gold, accepting strong WHO→ABPath fuzzy matches, auto-approving PathOut local tags, and suppressing generated lecture/textbook ontology junk by mapping or sequence inheritance.

Promotion is **enabled by default** but guarded by clear audits. It does not delete raw PDFs/videos or rebuild embeddings.

In [ ]:
# =========================
# Cell 1 — Configuration
# =========================
import os, json, re, sys, time, math, shutil, zipfile, hashlib, datetime, subprocess, sqlite3, glob, csv, html
from pathlib import Path
from datetime import timezone

PROJECT_ID = "pathology-annotation-project"
REGION = "us-central1"
LIVE_SERVICE = "pathology-hub-v04"
API_BASE_URL = "https://pathology-hub-v04-vorn5q2kga-uc.a.run.app"
HEALTH_URL = API_BASE_URL.rstrip('/') + "/health"
SEARCH_URL = API_BASE_URL.rstrip('/') + "/evidence/search"

# Promotion enabled by request. The promotion cell still aborts if clear audit fails.
PROMOTION_MODE = "backup_replace_live"   # options: "none", "backup_replace_live"
RESTART_CLOUD_RUN_AFTER_PROMOTION = True
RUN_API_PROOF = True

# Governance policy decisions from user.
AUTO_APPROVE_PATHOUT_LOCAL_TAGS = True
WHO_FUZZY_ACCEPT_THRESHOLD = 90
MAX_LECTURE_INHERIT_GAP_SEC = 600         # 10 minutes
MAX_LECTURE_INHERIT_ROW_GAP = 12
MAX_TEXTBOOK_INHERIT_PAGE_GAP = 2
MAX_TEXTBOOK_INHERIT_ROW_GAP = 25
HOLD_OFF_CURRICULUM_FACETS = True         # no morphology/IHC/etc facet generation in this pass

# Figure cleanup. Derived figure records only; raw PDFs are never deleted.
CLEAN_TEXTBOOK_FIGURES = True
PROBE_IMAGE_DIMENSIONS_WHEN_MISSING = True
MAX_IMAGE_DIMENSION_PROBES = 20000         # set None for all; large public figure maps can be slow
MIN_FIGURE_WIDTH = 40
MIN_FIGURE_HEIGHT = 40
MIN_FIGURE_AREA = 2500
MAX_ASPECT_RATIO = 12.0

# Optional: point to a GCS copy of ABPath tags. If blank, upload the ZIP when prompted.
ABSPATH_TAGS_GCS_URI = ""
ABSPATH_TAGS_LOCAL_CANDIDATES = [
    "/content/pathology_hub_abpath_source_tags.zip",
    "/content/pathology_hub_curriculum_tag_hardening_v11/inputs/pathology_hub_abpath_source_tags.zip",
    "/content/inputs/pathology_hub_abpath_source_tags.zip",
    "/content/drive/MyDrive/pathology_hub_abpath_source_tags.zip",
    "/mnt/data/pathology_hub_abpath_source_tags.zip",  # only works in this notebook-generation environment
]

# WHO source truth path supplied by user.
WHO_PROCESSED_GLOB = "gs://pathology-hub-0/WHO/WHO_JSON_PROCESSED/*.json"
WHO_EXTRA_URIS = []  # add explicit gs://... WHO files if needed

# Live backend-consumed metadata paths. These paths have been promoted in prior v2/v2.3/v10.3 work.
GCS = {
    # Textbooks
    "textbook_chunks_live": "gs://pathology_hub/02_normalized/textbooks/lean/tags/textbook_primary_tagged_chunks_v1.jsonl",
    "textbook_pages_live": "gs://pathology_hub/02_normalized/textbooks/lean/tags/textbook_primary_tagged_pages_v1.jsonl",
    "textbook_vector_docstore_live": "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_docstore.jsonl",
    "textbook_vector_manifest_live": "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_manifest.json",
    "textbook_figure_map_live": "gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN.jsonl",

    # PathOut AP-diagnostic vector subset
    "pathout_clean_pages_live": "gs://pathology_hub/02_normalized/pathology_outlines/pathout_allsite_v0_1/clean_pages/pathout_tagged_pages_AP_DIAGNOSTIC_v1.jsonl",
    "pathout_vector_docstore_live": "gs://pathology_hub/03_indexes/pathology_outlines/pathout_allsite_v0_1/vector_ap_diagnostic_v1/pathout_ap_diagnostic_vector_docstore.jsonl",
    "pathout_vector_manifest_live": "gs://pathology_hub/03_indexes/pathology_outlines/pathout_allsite_v0_1/vector_ap_diagnostic_v1/pathout_ap_diagnostic_vector_manifest.json",

    # Lectures/videos STRICT_CYTO v9 paths promoted to v10.3 content
    "lecture_tag_map_live": "gs://pathology_hub/02_normalized/lectures/tagging/lecture_primary_tag_map_STRICT_CYTO_v9.jsonl",
    "lecture_routed_chunks_live": "gs://pathology_hub/02_normalized/lectures/chunks/lecture_timecoded_tagged_chunks_ROUTED_ONLY_STRICT_CYTO_v9.jsonl",
    "lecture_vector_docstore_live": "gs://pathology_hub/03_indexes/lectures/vector_STRICT_CYTO_v9/lecture_timecoded_vector_docstore_STRICT_CYTO_v9.jsonl",
    "lecture_vector_manifest_live": "gs://pathology_hub/03_indexes/lectures/vector_STRICT_CYTO_v9/lecture_timecoded_vector_manifest_STRICT_CYTO_v9.json",
}

RUN_TS = datetime.datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
WORKDIR = Path(f"/content/pathology_hub_curriculum_tag_hardening_v11_{RUN_TS}")
IN_DIR = WORKDIR / "input"
OUT_DIR = WORKDIR / "output"
STAGE_DIR = OUT_DIR / "stage"
AUDIT_DIR = OUT_DIR / "audit"
for d in [WORKDIR, IN_DIR, OUT_DIR, STAGE_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

GCS_STAGE_PREFIX = f"gs://pathology_hub/02_normalized/tags/curriculum_hardening/v11/{RUN_TS}"
GCS_AUDIT_PREFIX = f"gs://pathology_hub/06_audits/tags/curriculum_hardening/v11/{RUN_TS}"
GCS_BACKUP_PREFIX = f"gs://pathology_hub/99_backups/curriculum_tag_hardening_v11/{RUN_TS}"
GCS_TAG_INDEX_LIVE = "gs://pathology_hub/03_indexes/tags/curriculum_hardening/v11/pathology_hub_approved_curriculum_tag_index_v11.sqlite"

print("WORKDIR:", WORKDIR)
print("Promotion mode:", PROMOTION_MODE)
print("GCS stage:", GCS_STAGE_PREFIX)

In [ ]:
# =========================
# Cell 2 — Auth, installs, and shell helper
# =========================
from google.colab import auth, files, userdata

auth.authenticate_user()

def now():
    return datetime.datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def sh(cmd, check=True, timeout=900):
    print(f"\n[{now()}] START: {cmd}", flush=True)
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
    elapsed = time.time() - t0
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    print(f"[{now()}] DONE rc={p.returncode} elapsed={elapsed:.1f}s", flush=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed rc={p.returncode}: {cmd}\n{p.stdout[-4000:] if p.stdout else ''}")
    return p

sh(f"gcloud config set project {PROJECT_ID}")

# rapidfuzz/Pillow are small and useful; install only if missing.
try:
    import rapidfuzz
except Exception:
    sh("pip -q install rapidfuzz")
try:
    from PIL import Image
except Exception:
    sh("pip -q install pillow")

from rapidfuzz import process, fuzz
from PIL import Image
import requests
API_KEY = userdata.get("X-API-Key")
print("API key available:", bool(API_KEY))

In [ ]:
# =========================
# Cell 3 — JSONL/GCS/text helper functions
# =========================

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_jsonl(path):
    out = []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                out.append(json.loads(line))
            except Exception as e:
                raise RuntimeError(f"Bad JSONL at {path}:{i+1}: {e}")
    return out

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, separators=(',', ':')) + '\n')

def read_json(path, default=None):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        if default is not None:
            return default
        raise

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def gcs_exists(uri):
    p = sh(f"gcloud storage ls {uri}", check=False, timeout=120)
    return p.returncode == 0

def gcs_cp(src, dst, check=True, timeout=1200):
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    return sh(f"gcloud storage cp {src} {dst}", check=check, timeout=timeout)

def gcs_cp_to_uri(src, dst_uri, check=True, timeout=1200):
    return sh(f"gcloud storage cp {src} {dst_uri}", check=check, timeout=timeout)

def gcs_ls(uri, check=False):
    p = sh(f"gcloud storage ls {uri}", check=check, timeout=300)
    if p.returncode != 0:
        return []
    return [x.strip() for x in p.stdout.splitlines() if x.strip().startswith('gs://')]

def download_live_artifact(key, required=True):
    uri = GCS[key]
    local = IN_DIR / (key + Path(uri).suffix)
    if not gcs_exists(uri):
        if required:
            raise FileNotFoundError(f"Missing required GCS artifact for {key}: {uri}")
        print("Optional artifact missing:", key, uri)
        return None
    gcs_cp(uri, local)
    return local

def get_nested(d, keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def first_nonempty(*vals):
    for v in vals:
        if v not in (None, '', [], {}):
            return v
    return None

def norm_text(s):
    s = '' if s is None else str(s)
    s = re.sub(r'[_\-/]+', ' ', s)
    s = re.sub(r'[^A-Za-z0-9]+', ' ', s).strip().lower()
    return re.sub(r'\s+', ' ', s)

def tag_root(tag):
    if not tag or tag == "__UNMAPPED__": return ""
    return str(tag).split('::')[0]

def tag_leaf(tag):
    if not tag: return ""
    return str(tag).split('::')[-1]

def tag_label(tag):
    return norm_text(tag.replace('::', ' '))

def primary_tag_of(r):
    return first_nonempty(r.get('primary_tag'), r.get('primary_tag_governed'), get_nested(r, ['metadata','primary_tag'])) or "__UNMAPPED__"

def set_primary_tag(r, tag):
    r['primary_tag'] = tag
    if isinstance(r.get('metadata'), dict):
        r['metadata']['primary_tag'] = tag
    return r

def record_id_of(r, fallback=None):
    return str(first_nonempty(r.get('record_id'), r.get('id'), r.get('chunk_id'), r.get('page_id'), r.get('url'), r.get('source_url'), fallback or ''))

def text_blob(r, max_chars=2500):
    parts = []
    for k in ['title','page_title','heading','section_title','slide_title','caption','diagnosis','entity','source_id','lecture_id','video_id','excerpt','text','body']:
        v = r.get(k)
        if v:
            parts.append(str(v))
    if isinstance(r.get('metadata'), dict):
        for k in ['title','page_title','heading','section_title','slide_title','caption','source_id','lecture_id','video_id']:
            v = r['metadata'].get(k)
            if v:
                parts.append(str(v))
    return '\n'.join(parts)[:max_chars]

JUNK_PATTERNS = [
    r'::Lectures::', r'::Textbooks::', r'\bPage[_\s-]*\d+\b', r'\bSlide[_\s-]*\d+\b',
    r'Digital[_\s-]*Pathology[_\s-]*Slide', r'Pathology[_\s-]*Slide', r'\bCase[_\s-]*\d+\b',
    r'\bError\b', r'\b\d+[_\s-]*seconds?[_\s-]*rule\b', r'\.svs\b', r'\.ndpi\b', r'\.jpg\b', r'\.png\b'
]
JUNK_RE = re.compile('|'.join(JUNK_PATTERNS), flags=re.I)

def is_junk_tag(tag):
    if not tag or tag == "__UNMAPPED__": return True
    return bool(JUNK_RE.search(str(tag)))

In [ ]:
# =========================
# Cell 4 — Load ABPath gold tags and WHO processed tags
# =========================

def extract_tag_strings_from_obj(obj):
    tags = []
    if isinstance(obj, str):
        if '::' in obj and len(obj) < 500:
            tags.append(obj.strip())
    elif isinstance(obj, list):
        for x in obj:
            tags.extend(extract_tag_strings_from_obj(x))
    elif isinstance(obj, dict):
        for k, v in obj.items():
            kl = str(k).lower()
            if any(tok in kl for tok in ['tag','primary_tag','diagnosis_tag','entity_tag','abpath']):
                tags.extend(extract_tag_strings_from_obj(v))
            elif isinstance(v, (dict, list)):
                tags.extend(extract_tag_strings_from_obj(v))
    return tags

def load_abpath_tags_from_zip(zip_path):
    tmp = WORKDIR / 'abpath_extract'
    if tmp.exists(): shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(tmp)
    tags = set()
    for p in tmp.rglob('*'):
        if p.is_dir(): continue
        try:
            txt = p.read_text(encoding='utf-8', errors='ignore')
        except Exception:
            continue
        if p.suffix.lower() in ['.json', '.jsonl']:
            if p.suffix.lower() == '.jsonl':
                for line in txt.splitlines():
                    if not line.strip(): continue
                    try:
                        tags.update(extract_tag_strings_from_obj(json.loads(line)))
                    except Exception:
                        pass
            else:
                try:
                    tags.update(extract_tag_strings_from_obj(json.loads(txt)))
                except Exception:
                    pass
        else:
            # CSV/TXT fallback: any cell/line containing ::
            for m in re.findall(r'[A-Za-z0-9_]+(?:::[A-Za-z0-9_]+)+', txt):
                tags.add(m.strip())
    tags = {t for t in tags if t and t != "__UNMAPPED__" and not is_junk_tag(t)}
    return sorted(tags)

abpath_zip = None
if ABSPATH_TAGS_GCS_URI:
    abpath_zip = IN_DIR / Path(ABSPATH_TAGS_GCS_URI).name
    gcs_cp(ABSPATH_TAGS_GCS_URI, abpath_zip)
else:
    for cand in ABSPATH_TAGS_LOCAL_CANDIDATES:
        if Path(cand).exists():
            abpath_zip = Path(cand)
            break
if abpath_zip is None:
    print("Upload pathology_hub_abpath_source_tags.zip")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("ABPath source tag ZIP is required.")
    abpath_zip = Path('/content') / next(iter(uploaded.keys()))

ABSPATH_GOLD_TAGS = load_abpath_tags_from_zip(abpath_zip)
if not ABSPATH_GOLD_TAGS:
    raise RuntimeError("No ABPath gold tags loaded.")
ABSPATH_SET = set(ABSPATH_GOLD_TAGS)
print("ABPath gold tags:", len(ABSPATH_GOLD_TAGS))
write_jsonl(OUT_DIR / 'abpath_gold_tags_v11.jsonl', [{'tag': t, 'tag_authority': 'abpath_gold'} for t in ABSPATH_GOLD_TAGS])

# WHO processed source tags.
who_dir = IN_DIR / 'who_processed_json'
who_dir.mkdir(exist_ok=True)
who_uris = gcs_ls(WHO_PROCESSED_GLOB) + WHO_EXTRA_URIS
if not who_uris:
    raise RuntimeError(f"No WHO processed JSON found at {WHO_PROCESSED_GLOB}")
for uri in who_uris:
    gcs_cp(uri, who_dir / Path(uri).name, check=True)

who_rows = []
for p in who_dir.glob('*.json'):
    try:
        obj = json.loads(p.read_text(encoding='utf-8', errors='ignore'))
    except Exception:
        continue
    tags = sorted(set(extract_tag_strings_from_obj(obj)))
    for t in tags:
        if t and t != "__UNMAPPED__" and not is_junk_tag(t):
            who_rows.append({'source': 'who', 'who_file': p.name, 'who_original_tag': t, 'record_id': f"who:{p.stem}:{hashlib.md5(t.encode()).hexdigest()[:12]}"})

# Fuzzy WHO→ABPath mapping. Exact or score >= 90 accepted.
gold_label_to_tags = {}
for t in ABSPATH_GOLD_TAGS:
    gold_label_to_tags.setdefault(tag_label(t), []).append(t)
gold_labels = list(gold_label_to_tags.keys())

who_mapped, who_review = [], []
for r in who_rows:
    t = r['who_original_tag']
    if t in ABPATH_SET:
        r.update({'canonical_tag': t, 'match_score': 100, 'match_basis': 'exact_abpath', 'tag_governance_status': 'who_exact_abpath_approved'})
        who_mapped.append(r)
    else:
        q = tag_label(t)
        match = process.extractOne(q, gold_labels, scorer=fuzz.token_set_ratio)
        if match and match[1] >= WHO_FUZZY_ACCEPT_THRESHOLD:
            canonical = gold_label_to_tags[match[0]][0]
            r.update({'canonical_tag': canonical, 'match_score': float(match[1]), 'match_basis': 'who_fuzzy_to_abpath_autoaccepted_>=90', 'tag_governance_status': 'who_fuzzy_abpath_approved'})
            who_mapped.append(r)
        else:
            r.update({'canonical_tag': None, 'match_score': float(match[1]) if match else None, 'match_basis': 'who_fuzzy_below_threshold', 'tag_governance_status': 'who_needs_review_excluded_from_tag_index'})
            who_review.append(r)

print("WHO tag-bearing rows:", len(who_rows))
print("WHO mapped/approved:", len(who_mapped))
print("WHO needs review/excluded:", len(who_review))
write_jsonl(AUDIT_DIR / 'WHO_TO_ABPATH_FUZZY_ACCEPTED_v11.jsonl', who_mapped)
write_jsonl(AUDIT_DIR / 'WHO_TO_ABPATH_NEEDS_REVIEW_v11.jsonl', who_review)

In [ ]:
# =========================
# Cell 5 — Download current live metadata artifacts
# =========================
required_keys = [
    'textbook_chunks_live', 'textbook_vector_docstore_live', 'textbook_vector_manifest_live',
    'pathout_vector_docstore_live', 'pathout_vector_manifest_live',
    'lecture_tag_map_live', 'lecture_routed_chunks_live', 'lecture_vector_docstore_live', 'lecture_vector_manifest_live'
]
optional_keys = ['textbook_pages_live', 'pathout_clean_pages_live', 'textbook_figure_map_live']
local_artifacts = {}
for k in required_keys:
    local_artifacts[k] = download_live_artifact(k, required=True)
for k in optional_keys:
    local_artifacts[k] = download_live_artifact(k, required=False)

print("Reading JSONL artifacts. This may take a few minutes.")
textbook_chunks = read_jsonl(local_artifacts['textbook_chunks_live'])
textbook_vector_docstore = read_jsonl(local_artifacts['textbook_vector_docstore_live'])
textbook_manifest = read_json(local_artifacts['textbook_vector_manifest_live'], default={})
pathout_docstore = read_jsonl(local_artifacts['pathout_vector_docstore_live'])
pathout_manifest = read_json(local_artifacts['pathout_vector_manifest_live'], default={})
lecture_tag_map = read_jsonl(local_artifacts['lecture_tag_map_live'])
lecture_routed_chunks = read_jsonl(local_artifacts['lecture_routed_chunks_live'])
lecture_docstore = read_jsonl(local_artifacts['lecture_vector_docstore_live'])
lecture_manifest = read_json(local_artifacts['lecture_vector_manifest_live'], default={})
textbook_pages = read_jsonl(local_artifacts['textbook_pages_live']) if local_artifacts.get('textbook_pages_live') else []
pathout_clean_pages = read_jsonl(local_artifacts['pathout_clean_pages_live']) if local_artifacts.get('pathout_clean_pages_live') else []
textbook_figure_map = read_jsonl(local_artifacts['textbook_figure_map_live']) if local_artifacts.get('textbook_figure_map_live') else []

print(json.dumps({
    'textbook_chunks': len(textbook_chunks),
    'textbook_vector_docstore': len(textbook_vector_docstore),
    'pathout_docstore': len(pathout_docstore),
    'lecture_docstore': len(lecture_docstore),
    'lecture_routed_chunks': len(lecture_routed_chunks),
    'lecture_tag_map': len(lecture_tag_map),
    'textbook_figure_map': len(textbook_figure_map),
    'who_mapped_rows': len(who_mapped),
}, indent=2))

In [ ]:
# =========================
# Cell 6 — Build approved tag universe
# =========================
approved_tags = set(ABSPATH_SET)
tag_authority = {t: 'abpath_gold' for t in ABPATH_SET}

# WHO accepted tags map to ABPath canonical tags.
for r in who_mapped:
    t = r['canonical_tag']
    if t:
        approved_tags.add(t)
        tag_authority.setdefault(t, 'who_mapped_abpath')

# PathOut local tags auto-approved by user decision.
pathout_tags = set()
for r in pathout_docstore + pathout_clean_pages:
    t = primary_tag_of(r)
    if t and t != "__UNMAPPED__" and not is_junk_tag(t):
        pathout_tags.add(t)
if AUTO_APPROVE_PATHOUT_LOCAL_TAGS:
    for t in pathout_tags:
        approved_tags.add(t)
        tag_authority.setdefault(t, 'approved_pathout_local')

# Index approved tags by root/labels for fast matching.
approved_by_root = {}
for t in approved_tags:
    approved_by_root.setdefault(tag_root(t), []).append(t)

approved_label_by_root = {}
approved_label_to_tag_by_root = {}
for root, tags in approved_by_root.items():
    labels = []
    mp = {}
    for t in sorted(tags):
        lab = tag_label(t)
        labels.append(lab)
        mp.setdefault(lab, t)
    approved_label_by_root[root] = labels
    approved_label_to_tag_by_root[root] = mp

print(json.dumps({
    'approved_total_unique_tags': len(approved_tags),
    'abpath_gold': len(ABSPATH_SET),
    'who_mapped_abpath_rows': len(who_mapped),
    'pathout_local_approved_unique_tags': len(pathout_tags),
    'roots': sorted(list(approved_by_root))[:50],
}, indent=2))

def approved_tag_status(t):
    if not t or t == "__UNMAPPED__":
        return False, 'unmapped'
    if is_junk_tag(t):
        return False, 'junk_pattern'
    if t in approved_tags:
        return True, tag_authority.get(t, 'approved')
    return False, 'not_approved'

# Current tag mapping cache: map non-approved but meaningful tags to approved tags only if very close.
tag_mapping_cache = {}
def map_tag_to_approved_candidate(tag, root_hint=None, min_score=94):
    if not tag or tag == "__UNMAPPED__":
        return None, None, 'no_tag'
    ok, basis = approved_tag_status(tag)
    if ok:
        return tag, 100, basis
    if is_junk_tag(tag):
        return None, None, 'junk_no_fuzzy'
    key = (tag, root_hint, min_score)
    if key in tag_mapping_cache:
        return tag_mapping_cache[key]
    root = root_hint or tag_root(tag)
    labels = approved_label_by_root.get(root, [])
    if not labels:
        tag_mapping_cache[key] = (None, None, 'no_root_catalog')
        return tag_mapping_cache[key]
    q = tag_label(tag)
    match = process.extractOne(q, labels, scorer=fuzz.token_set_ratio)
    if match and match[1] >= min_score:
        mapped = approved_label_to_tag_by_root[root][match[0]]
        tag_mapping_cache[key] = (mapped, float(match[1]), 'fuzzy_current_tag_to_approved')
    else:
        tag_mapping_cache[key] = (None, float(match[1]) if match else None, 'fuzzy_below_threshold')
    return tag_mapping_cache[key]

In [ ]:
# =========================
# Cell 7 — Sequence helpers and governance functions
# =========================

def parse_lecture_id(r):
    for k in ['lecture_id', 'video_id', 'source_id']:
        v = first_nonempty(r.get(k), get_nested(r, ['metadata', k]))
        if v: return str(v)
    txt = text_blob(r, 1000)
    m = re.search(r'Lecture:\s*([^\n\r]+)', txt, re.I)
    if m: return m.group(1).strip()
    return str(first_nonempty(r.get('title'), get_nested(r, ['metadata','title']), 'UNKNOWN_LECTURE'))

def parse_time_sec(r):
    for k in ['start_sec', 'start_seconds', 'start_time', 'start']:
        v = first_nonempty(r.get(k), get_nested(r, ['metadata', k]))
        if v is not None:
            try: return float(v)
            except Exception: pass
    txt = text_blob(r, 1200)
    m = re.search(r'Time:\s*([0-9]+(?:\.[0-9]+)?)\s*[–\-]\s*([0-9]+(?:\.[0-9]+)?)\s*sec', txt, re.I)
    if m:
        return float(m.group(1))
    return None

def textbook_source_id(r):
    for k in ['source_id','book_id','source','pdf_id']:
        v = first_nonempty(r.get(k), get_nested(r, ['metadata', k]))
        if v: return str(v)
    return str(first_nonempty(r.get('source_name'), get_nested(r, ['metadata','source_name']), 'UNKNOWN_TEXTBOOK'))

def textbook_page_num(r):
    for k in ['page', 'page_number', 'pdf_page', 'page_index']:
        v = first_nonempty(r.get(k), get_nested(r, ['metadata', k]))
        if v is not None:
            try: return int(float(str(v).strip()))
            except Exception: pass
    pk = first_nonempty(r.get('page_key'), get_nested(r, ['metadata', 'page_key']))
    if pk:
        m = re.search(r'(?:page|p)[_\- ]*(\d+)', str(pk), re.I)
        if m: return int(m.group(1))
    return None

def sort_index(r, fallback):
    for k in ['chunk_index', 'row_index', 'docstore_row_index', 'segment_index', 'order']:
        v = first_nonempty(r.get(k), get_nested(r, ['metadata', k]))
        if v is not None:
            try: return int(float(str(v)))
            except Exception: pass
    return fallback

def govern_record_direct_or_inherit(r, last_state, source_family, row_i, root_hint=None, sequence_key=None, position=None, max_gap=None, max_row_gap=None):
    original = primary_tag_of(r)
    original_root = root_hint or tag_root(original)
    mapped, score, basis = map_tag_to_approved_candidate(original, original_root)
    out = dict(r)
    out['primary_tag_original_pre_curriculum_v11'] = original
    out['tag_governance_version'] = 'curriculum_hardening_v11'

    if mapped:
        governed = mapped
        status = 'approved_direct' if score == 100 else 'approved_fuzzy_current_tag'
        new_last = {'tag': governed, 'row_i': row_i, 'position': position, 'record_id': record_id_of(r, row_i), 'source_family': source_family}
    else:
        inherited = False
        inherit_from = None
        if last_state and sequence_key == last_state.get('sequence_key'):
            row_gap_ok = (max_row_gap is None) or (row_i - last_state.get('row_i', row_i) <= max_row_gap)
            pos_gap_ok = True
            if max_gap is not None and position is not None and last_state.get('position') is not None:
                pos_gap_ok = abs(position - last_state['position']) <= max_gap
            if row_gap_ok and pos_gap_ok:
                inherited = True
                inherit_from = last_state
        if inherited:
            governed = inherit_from['tag']
            status = 'inherited_context'
            basis = f"weak_or_junk_current_tag_inherited_from_prior_{source_family}_context"
            new_last = last_state
        else:
            governed = "__UNMAPPED__"
            status = 'unmapped_no_prior_context'
            basis = basis or 'no_approved_mapping_no_prior_context'
            new_last = last_state

    set_primary_tag(out, governed)
    out['primary_tag_governed'] = governed
    out['tag_governance_status'] = status
    out['tag_governance_basis'] = basis
    if score is not None: out['tag_governance_match_score'] = score
    if status == 'inherited_context' and last_state:
        out['inherited_from_record_id'] = last_state.get('record_id')
        out['inherited_from_tag'] = last_state.get('tag')
        out['inheritance_distance_rows'] = row_i - last_state.get('row_i', row_i)
        if position is not None and last_state.get('position') is not None:
            out['inheritance_distance_position'] = position - last_state.get('position')
    return out, new_last

# PathOut governance: auto-approve local PathOut tags unless junk/unmapped.
def govern_pathout_rows(rows, source_label):
    out, counts = [], {'approved_pathout_local': 0, 'unmapped_or_excluded': 0}
    for i, r in enumerate(rows):
        rr = dict(r)
        t = primary_tag_of(rr)
        rr['primary_tag_original_pre_curriculum_v11'] = t
        rr['tag_governance_version'] = 'curriculum_hardening_v11'
        if t and t != "__UNMAPPED__" and not is_junk_tag(t):
            rr['primary_tag_governed'] = t
            rr['tag_governance_status'] = 'approved_pathout_local'
            rr['tag_governance_basis'] = 'pathout_entity_tag_auto_approved_by_policy'
            set_primary_tag(rr, t)
            counts['approved_pathout_local'] += 1
        else:
            rr['primary_tag_governed'] = "__UNMAPPED__"
            rr['tag_governance_status'] = 'pathout_unmapped_or_junk_excluded'
            rr['tag_governance_basis'] = 'no_valid_pathout_tag'
            set_primary_tag(rr, "__UNMAPPED__")
            counts['unmapped_or_excluded'] += 1
        out.append(rr)
    return out, counts

In [ ]:
# =========================
# Cell 8 — Apply governance: PathOut, lectures, textbooks
# =========================
# PathOut
pathout_docstore_gov, pathout_doc_counts = govern_pathout_rows(pathout_docstore, 'pathout_docstore')
pathout_clean_pages_gov, pathout_clean_counts = govern_pathout_rows(pathout_clean_pages, 'pathout_clean_pages') if pathout_clean_pages else ([], {})

# Lectures: canonical sequence is vector docstore.
lecture_groups = {}
for i, r in enumerate(lecture_docstore):
    seq = parse_lecture_id(r)
    pos = parse_time_sec(r)
    lecture_groups.setdefault(seq, []).append((i, r, pos))

lecture_docstore_gov = [None] * len(lecture_docstore)
lecture_decision_by_chunk = {}
lecture_audit_examples = []
for seq, items in lecture_groups.items():
    items_sorted = sorted(items, key=lambda x: (x[2] if x[2] is not None else 1e18, sort_index(x[1], x[0]), x[0]))
    last = None
    if last: last['sequence_key'] = seq
    for local_order, (i, r, pos) in enumerate(items_sorted):
        root_hint = tag_root(primary_tag_of(r))
        governed, new_last = govern_record_direct_or_inherit(
            r, last, 'lecture', local_order, root_hint=root_hint, sequence_key=seq, position=pos,
            max_gap=MAX_LECTURE_INHERIT_GAP_SEC, max_row_gap=MAX_LECTURE_INHERIT_ROW_GAP
        )
        if new_last is not None:
            new_last['sequence_key'] = seq
        if governed.get('tag_governance_status') in ['approved_direct', 'approved_fuzzy_current_tag']:
            last = new_last
            last['sequence_key'] = seq
        elif new_last is not None and new_last is last:
            pass
        lecture_docstore_gov[i] = governed
        ck = first_nonempty(governed.get('chunk_id'), governed.get('id'), governed.get('record_id'))
        if ck:
            lecture_decision_by_chunk[str(ck)] = {
                'primary_tag': governed['primary_tag_governed'],
                'status': governed['tag_governance_status'],
                'basis': governed['tag_governance_basis'],
                'inherited_from_record_id': governed.get('inherited_from_record_id'),
            }
        if len(lecture_audit_examples) < 100 and governed['tag_governance_status'] in ['inherited_context','unmapped_no_prior_context']:
            lecture_audit_examples.append({
                'sequence': seq, 'row': i, 'original': governed.get('primary_tag_original_pre_curriculum_v11'),
                'governed': governed.get('primary_tag_governed'), 'status': governed.get('tag_governance_status'),
                'basis': governed.get('tag_governance_basis')
            })

# Apply lecture decisions to routed chunks/tag map by chunk_id where possible; otherwise direct govern in file order.
def apply_lecture_decisions(rows, label):
    out = []
    last_by_seq = {}
    for i, r in enumerate(rows):
        rr = dict(r)
        ck = str(first_nonempty(rr.get('chunk_id'), rr.get('id'), rr.get('record_id'), ''))
        if ck in lecture_decision_by_chunk:
            dec = lecture_decision_by_chunk[ck]
            orig = primary_tag_of(rr)
            rr['primary_tag_original_pre_curriculum_v11'] = orig
            set_primary_tag(rr, dec['primary_tag'])
            rr['primary_tag_governed'] = dec['primary_tag']
            rr['tag_governance_version'] = 'curriculum_hardening_v11'
            rr['tag_governance_status'] = dec['status']
            rr['tag_governance_basis'] = 'applied_from_vector_docstore_sequence_decision_' + dec['basis']
            if dec.get('inherited_from_record_id'):
                rr['inherited_from_record_id'] = dec['inherited_from_record_id']
        else:
            seq = parse_lecture_id(rr)
            pos = parse_time_sec(rr)
            root_hint = tag_root(primary_tag_of(rr))
            governed, new_last = govern_record_direct_or_inherit(rr, last_by_seq.get(seq), label, i, root_hint=root_hint, sequence_key=seq, position=pos, max_gap=MAX_LECTURE_INHERIT_GAP_SEC, max_row_gap=MAX_LECTURE_INHERIT_ROW_GAP)
            if governed.get('tag_governance_status') in ['approved_direct', 'approved_fuzzy_current_tag']:
                new_last['sequence_key'] = seq
                last_by_seq[seq] = new_last
            rr = governed
        out.append(rr)
    return out

lecture_routed_chunks_gov = apply_lecture_decisions(lecture_routed_chunks, 'lecture_routed_chunks')
lecture_tag_map_gov = apply_lecture_decisions(lecture_tag_map, 'lecture_tag_map')

# Textbooks: canonical sequence is tagged chunks.
textbook_groups = {}
for i, r in enumerate(textbook_chunks):
    seq = textbook_source_id(r)
    page = textbook_page_num(r)
    textbook_groups.setdefault(seq, []).append((i, r, page))

textbook_chunks_gov = [None] * len(textbook_chunks)
textbook_decision_by_chunk = {}
textbook_decision_by_page_key = {}
textbook_audit_examples = []
for seq, items in textbook_groups.items():
    items_sorted = sorted(items, key=lambda x: (x[2] if x[2] is not None else 1e18, sort_index(x[1], x[0]), x[0]))
    last = None
    for local_order, (i, r, page) in enumerate(items_sorted):
        root_hint = tag_root(primary_tag_of(r))
        governed, new_last = govern_record_direct_or_inherit(
            r, last, 'textbook', local_order, root_hint=root_hint, sequence_key=seq, position=page,
            max_gap=MAX_TEXTBOOK_INHERIT_PAGE_GAP, max_row_gap=MAX_TEXTBOOK_INHERIT_ROW_GAP
        )
        if new_last is not None:
            new_last['sequence_key'] = seq
        if governed.get('tag_governance_status') in ['approved_direct', 'approved_fuzzy_current_tag']:
            last = new_last
            last['sequence_key'] = seq
        textbook_chunks_gov[i] = governed
        ck = first_nonempty(governed.get('chunk_id'), governed.get('id'), governed.get('record_id'))
        if ck:
            textbook_decision_by_chunk[str(ck)] = {
                'primary_tag': governed['primary_tag_governed'], 'status': governed['tag_governance_status'], 'basis': governed['tag_governance_basis'],
                'inherited_from_record_id': governed.get('inherited_from_record_id')
            }
        pk = first_nonempty(governed.get('page_key'), get_nested(governed, ['metadata', 'page_key']))
        if pk and governed['primary_tag_governed'] != "__UNMAPPED__":
            textbook_decision_by_page_key[str(pk)] = textbook_decision_by_chunk.get(str(ck), {'primary_tag': governed['primary_tag_governed'], 'status': governed['tag_governance_status'], 'basis': governed['tag_governance_basis']})
        if len(textbook_audit_examples) < 100 and governed['tag_governance_status'] in ['inherited_context','unmapped_no_prior_context']:
            textbook_audit_examples.append({'source': seq, 'row': i, 'page': page, 'original': governed.get('primary_tag_original_pre_curriculum_v11'), 'governed': governed.get('primary_tag_governed'), 'status': governed.get('tag_governance_status')})

def apply_textbook_decisions(rows, label):
    out = []
    for i, r in enumerate(rows):
        rr = dict(r)
        ck = str(first_nonempty(rr.get('chunk_id'), rr.get('id'), rr.get('record_id'), ''))
        pk = str(first_nonempty(rr.get('page_key'), get_nested(rr, ['metadata','page_key']), ''))
        dec = textbook_decision_by_chunk.get(ck) or textbook_decision_by_page_key.get(pk)
        if dec:
            orig = primary_tag_of(rr)
            rr['primary_tag_original_pre_curriculum_v11'] = orig
            set_primary_tag(rr, dec['primary_tag'])
            rr['primary_tag_governed'] = dec['primary_tag']
            rr['tag_governance_version'] = 'curriculum_hardening_v11'
            rr['tag_governance_status'] = dec['status']
            rr['tag_governance_basis'] = 'applied_from_textbook_chunk_sequence_decision_' + dec['basis']
            if dec.get('inherited_from_record_id'):
                rr['inherited_from_record_id'] = dec['inherited_from_record_id']
        else:
            orig = primary_tag_of(rr)
            mapped, score, basis = map_tag_to_approved_candidate(orig, tag_root(orig))
            rr['primary_tag_original_pre_curriculum_v11'] = orig
            set_primary_tag(rr, mapped or "__UNMAPPED__")
            rr['primary_tag_governed'] = mapped or "__UNMAPPED__"
            rr['tag_governance_version'] = 'curriculum_hardening_v11'
            rr['tag_governance_status'] = 'approved_direct_or_fuzzy' if mapped else 'unmapped_no_chunk_context'
            rr['tag_governance_basis'] = basis
        out.append(rr)
    return out

textbook_vector_docstore_gov = apply_textbook_decisions(textbook_vector_docstore, 'textbook_vector_docstore')
textbook_pages_gov = apply_textbook_decisions(textbook_pages, 'textbook_pages') if textbook_pages else []

print("Governance applied.")

In [ ]:
# =========================
# Cell 9 — Strict derived textbook figure cleanup
# =========================

def parse_dim_val(v):
    try:
        if v is None: return None
        return int(float(str(v).strip()))
    except Exception:
        return None

def figure_dims_from_record(r):
    candidates_w = ['width','image_width','w','pixel_width','img_width']
    candidates_h = ['height','image_height','h','pixel_height','img_height']
    w = first_nonempty(*[r.get(k) for k in candidates_w], *[get_nested(r, ['metadata', k]) for k in candidates_w])
    h = first_nonempty(*[r.get(k) for k in candidates_h], *[get_nested(r, ['metadata', k]) for k in candidates_h])
    w, h = parse_dim_val(w), parse_dim_val(h)
    if w and h: return w, h, 'metadata'
    dim = first_nonempty(r.get('dimensions'), get_nested(r, ['metadata','dimensions']))
    if dim:
        m = re.search(r'(\d+)\s*[x×]\s*(\d+)', str(dim))
        if m: return int(m.group(1)), int(m.group(2)), 'metadata_dimensions_string'
    return None, None, 'missing'

def image_url_of(r):
    return first_nonempty(r.get('image_url'), r.get('figure_url'), r.get('public_url'), r.get('url'), get_nested(r, ['metadata','image_url']), get_nested(r, ['metadata','figure_url']))

def probe_image_dims(url):
    try:
        resp = requests.get(url, timeout=12, stream=True)
        resp.raise_for_status()
        img = Image.open(resp.raw)
        return img.size[0], img.size[1], 'probed'
    except Exception as e:
        return None, None, 'probe_failed:' + str(e)[:120]

def is_bad_figure_dim(w, h):
    if not w or not h: return False, 'missing_dimensions_not_excluded'
    area = w*h
    ratio = max(w/h, h/w) if w and h else 999
    if w < MIN_FIGURE_WIDTH: return True, f'width<{MIN_FIGURE_WIDTH}'
    if h < MIN_FIGURE_HEIGHT: return True, f'height<{MIN_FIGURE_HEIGHT}'
    if area < MIN_FIGURE_AREA: return True, f'area<{MIN_FIGURE_AREA}'
    if ratio > MAX_ASPECT_RATIO: return True, f'aspect_ratio>{MAX_ASPECT_RATIO}'
    return False, 'kept_dimensions_ok'

textbook_figure_map_gov = []
figure_exclusions = []
probe_count = 0
if CLEAN_TEXTBOOK_FIGURES and textbook_figure_map:
    for i, r in enumerate(textbook_figure_map):
        rr = dict(r)
        w, h, dim_basis = figure_dims_from_record(rr)
        if (not w or not h) and PROBE_IMAGE_DIMENSIONS_WHEN_MISSING and (MAX_IMAGE_DIMENSION_PROBES is None or probe_count < MAX_IMAGE_DIMENSION_PROBES):
            url = image_url_of(rr)
            if url and str(url).startswith('http'):
                w, h, dim_basis = probe_image_dims(url)
                probe_count += 1
        bad, reason = is_bad_figure_dim(w, h)
        if bad:
            ex = {'row_index': i, 'reason': reason, 'width': w, 'height': h, 'dim_basis': dim_basis, 'image_url': image_url_of(rr), 'source_id': rr.get('source_id'), 'page': rr.get('page')}
            figure_exclusions.append(ex)
        else:
            rr['figure_governance_version'] = 'curriculum_hardening_v11'
            rr['figure_governance_status'] = 'kept'
            rr['figure_dimension_basis'] = dim_basis
            if w: rr['governed_width'] = w
            if h: rr['governed_height'] = h
            textbook_figure_map_gov.append(rr)
else:
    textbook_figure_map_gov = textbook_figure_map

print(json.dumps({'figure_input': len(textbook_figure_map), 'figure_kept': len(textbook_figure_map_gov), 'figure_excluded': len(figure_exclusions), 'dimension_probes': probe_count}, indent=2))
write_jsonl(AUDIT_DIR / 'TEXTBOOK_FIGURE_EXCLUSIONS_v11.jsonl', figure_exclusions)

In [ ]:
# =========================
# Cell 10 — Build approved-only tag index, browser, manifests, and audits
# =========================

def governed_tag_of(r):
    return first_nonempty(r.get('primary_tag_governed'), r.get('primary_tag')) or "__UNMAPPED__"

def visible_tag(t):
    return bool(t and t != "__UNMAPPED__" and not is_junk_tag(t) and t in approved_tags)

def summarize_rows(rows, label):
    c = {'total': len(rows), 'mapped_visible': 0, 'unmapped': 0, 'junk_visible': 0, 'status_counts': {}}
    for r in rows:
        t = governed_tag_of(r)
        st = r.get('tag_governance_status', 'missing_status')
        c['status_counts'][st] = c['status_counts'].get(st, 0) + 1
        if t == "__UNMAPPED__": c['unmapped'] += 1
        elif is_junk_tag(t): c['junk_visible'] += 1
        else: c['mapped_visible'] += 1
    return c

summaries = {
    'textbook_chunks': summarize_rows(textbook_chunks_gov, 'textbook_chunks'),
    'textbook_vector_docstore': summarize_rows(textbook_vector_docstore_gov, 'textbook_vector_docstore'),
    'lecture_docstore': summarize_rows(lecture_docstore_gov, 'lecture_docstore'),
    'lecture_routed_chunks': summarize_rows(lecture_routed_chunks_gov, 'lecture_routed_chunks'),
    'lecture_tag_map': summarize_rows(lecture_tag_map_gov, 'lecture_tag_map'),
    'pathout_docstore': summarize_rows(pathout_docstore_gov, 'pathout_docstore'),
}

# Hard clear-audit gates.
forbidden_examples = []
for source_label, rows in [('textbook_chunks', textbook_chunks_gov), ('textbook_vector_docstore', textbook_vector_docstore_gov), ('lecture_docstore', lecture_docstore_gov), ('lecture_routed_chunks', lecture_routed_chunks_gov), ('lecture_tag_map', lecture_tag_map_gov)]:
    for i, r in enumerate(rows):
        t = governed_tag_of(r)
        if t != "__UNMAPPED__" and is_junk_tag(t):
            forbidden_examples.append({'source': source_label, 'row_index': i, 'primary_tag': t, 'record_id': record_id_of(r, i)})
            if len(forbidden_examples) >= 50: break
    if len(forbidden_examples) >= 50: break

# Build SQLite approved-only index.
index_path = OUT_DIR / 'pathology_hub_approved_curriculum_tag_index_v11.sqlite'
if index_path.exists(): index_path.unlink()
conn = sqlite3.connect(index_path)
cur = conn.cursor()
cur.execute("""CREATE TABLE records (
    source TEXT, source_family TEXT, record_id TEXT, primary_tag TEXT, tag_root TEXT,
    tag_authority TEXT, governance_status TEXT, title TEXT, url TEXT, excerpt TEXT,
    normalized_artifact_gcs_uri TEXT, row_index INTEGER, locator_json TEXT
)""")
cur.execute('CREATE INDEX idx_records_tag ON records(primary_tag)')
cur.execute('CREATE INDEX idx_records_root ON records(tag_root)')
cur.execute('CREATE INDEX idx_records_source ON records(source)')
cur.execute("""CREATE TABLE tags (
    primary_tag TEXT PRIMARY KEY, tag_root TEXT, tag_authority TEXT, record_count INTEGER
)""")

def insert_rows_to_index(rows, source, source_family, artifact_uri):
    inserted = 0
    for i, r in enumerate(rows):
        t = governed_tag_of(r)
        if not visible_tag(t):
            continue
        title = first_nonempty(r.get('title'), r.get('page_title'), r.get('heading'), r.get('slide_title'), get_nested(r, ['metadata','title']), '')
        url = first_nonempty(r.get('source_url'), r.get('url'), r.get('source_page_url'), r.get('video_time_url'), r.get('video_url'), get_nested(r, ['metadata','url']), '')
        excerpt = first_nonempty(r.get('excerpt'), r.get('text'), '')
        if excerpt: excerpt = str(excerpt)[:1000]
        locator = {k: r.get(k) for k in ['source_id','page','page_key','chunk_id','video_id','lecture_id','start_sec','end_sec'] if r.get(k) is not None}
        cur.execute('INSERT INTO records VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)', (
            source, source_family, record_id_of(r, i), t, tag_root(t), tag_authority.get(t, 'approved_local'), r.get('tag_governance_status',''),
            str(title)[:500], str(url)[:1000], str(excerpt)[:1000], artifact_uri, i, json.dumps(locator, ensure_ascii=False)
        ))
        inserted += 1
    return inserted

insert_counts = {}
insert_counts['textbooks'] = insert_rows_to_index(textbook_vector_docstore_gov, 'textbooks', 'textbook_vector_docstore', GCS['textbook_vector_docstore_live'])
insert_counts['pathout'] = insert_rows_to_index(pathout_docstore_gov, 'pathout', 'pathout_ap_diagnostic_vector_docstore', GCS['pathout_vector_docstore_live'])
insert_counts['lectures'] = insert_rows_to_index(lecture_docstore_gov, 'lectures', 'lecture_vector_docstore', GCS['lecture_vector_docstore_live'])
insert_counts['videos'] = 0  # same lecture records are queryable via videos in API; avoid duplicate index rows.
# WHO rows from processed source.
for i, r in enumerate(who_mapped):
    t = r.get('canonical_tag')
    if not visible_tag(t): continue
    cur.execute('INSERT INTO records VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)', (
        'who', 'who_processed_json', r.get('record_id'), t, tag_root(t), 'who_mapped_abpath', r.get('tag_governance_status'),
        Path(r.get('who_file','')).stem, '', '', WHO_PROCESSED_GLOB, i, json.dumps({'who_file': r.get('who_file'), 'who_original_tag': r.get('who_original_tag'), 'match_score': r.get('match_score')}, ensure_ascii=False)
    ))
insert_counts['who'] = cur.execute("SELECT COUNT(*) FROM records WHERE source='who'").fetchone()[0]

# Fill tags table from records.
for t, n in cur.execute('SELECT primary_tag, COUNT(*) FROM records GROUP BY primary_tag').fetchall():
    cur.execute('INSERT OR REPLACE INTO tags VALUES (?,?,?,?)', (t, tag_root(t), tag_authority.get(t, 'approved_local'), n))
conn.commit(); conn.close()

# Static browser from approved index tags.
conn = sqlite3.connect(index_path)
tag_rows = conn.execute('SELECT primary_tag, tag_root, tag_authority, record_count FROM tags ORDER BY primary_tag').fetchall()
conn.close()

def make_tree(tags):
    root = {}
    for t, r, auth, n in tags:
        node = root
        parts = t.split('::')
        for p in parts:
            node = node.setdefault(p, {})
        node['_meta'] = {'tag': t, 'authority': auth, 'record_count': n}
    return root

tag_tree = make_tree(tag_rows)
write_json(OUT_DIR / 'approved_curriculum_tag_tree_v11.json', tag_tree)
with open(OUT_DIR / 'approved_curriculum_tag_catalog_v11.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['primary_tag','root','tag_authority','record_count'])
    w.writerows(tag_rows)

browser_html = f"""<!doctype html><html><head><meta charset='utf-8'><title>Pathology Hub Approved Curriculum Tags v11</title>
<style>body{{font-family:system-ui,Arial;margin:20px;}} details{{margin-left:1em}} summary{{cursor:pointer}} .tag{{font-family:monospace}} .bad{{color:#b00}} input{{width:420px;padding:6px}}</style></head><body>
<h1>Approved Curriculum Tags v11</h1><p>Approved-only tag index. Generated tags containing lecture/textbook/page/slide artifacts are excluded. Records in index: {sum(insert_counts.values())}. Unique tags: {len(tag_rows)}.</p>
<input id='q' placeholder='Filter tags...' oninput='filterTags()'><div id='tree'></div>
<script>const tree={json.dumps(tag_tree)};
function esc(s){{return String(s).replace(/[&<>]/g,c=>({{'&':'&amp;','<':'&lt;','>':'&gt;'}}[c]));}}
function renderNode(obj,name=''){{let html=''; for(const [k,v] of Object.entries(obj)){{if(k==='_meta')continue; const meta=v._meta; const label=meta?`${{k}} <span class="tag">(${{meta.record_count}}; ${{meta.authority}})</span>`:k; html+=`<details open data-name="${{esc(k.toLowerCase())}}"><summary>${{label}}</summary>${{meta?`<div class="tag">${{esc(meta.tag)}}</div>`:''}}${{renderNode(v,k)}}</details>`;}} return html;}}
document.getElementById('tree').innerHTML=renderNode(tree);
function filterTags(){{const q=document.getElementById('q').value.toLowerCase(); document.querySelectorAll('details').forEach(d=>{{d.style.display=!q||d.textContent.toLowerCase().includes(q)?'':'none';}})}}
</script></body></html>"""
(OUT_DIR / 'approved_curriculum_tag_browser_v11.html').write_text(browser_html, encoding='utf-8')

clear_audit = {
    'schema_version': 'pathology_hub_curriculum_tag_hardening_clear_audit.v11',
    'generated_at_utc': now(),
    'summaries': summaries,
    'approved_index_insert_counts': insert_counts,
    'approved_index_unique_tags': len(tag_rows),
    'approved_index_path': str(index_path),
    'who_original_unique_tags': len({r['who_original_tag'] for r in who_rows}),
    'who_mapped_accepted_rows': len(who_mapped),
    'who_needs_review_rows': len(who_review),
    'pathout_local_approved_unique_tags': len(pathout_tags),
    'forbidden_governed_primary_tag_examples': forbidden_examples,
    'figure_cleanup': {'input': len(textbook_figure_map), 'kept': len(textbook_figure_map_gov), 'excluded': len(figure_exclusions)},
    'clear_audit_passed': len(forbidden_examples) == 0 and len(tag_rows) > 0,
    'policy': {
        'auto_approve_pathout_local_tags': AUTO_APPROVE_PATHOUT_LOCAL_TAGS,
        'who_fuzzy_accept_threshold': WHO_FUZZY_ACCEPT_THRESHOLD,
        'max_lecture_inherit_gap_sec': MAX_LECTURE_INHERIT_GAP_SEC,
        'max_textbook_inherit_page_gap': MAX_TEXTBOOK_INHERIT_PAGE_GAP,
        'hold_off_curriculum_facets': HOLD_OFF_CURRICULUM_FACETS,
    }
}
write_json(AUDIT_DIR / 'CURRICULUM_TAG_HARDENING_V11_CLEAR_AUDIT.json', clear_audit)
write_jsonl(AUDIT_DIR / 'LECTURE_INHERITANCE_EXAMPLES_v11.jsonl', lecture_audit_examples)
write_jsonl(AUDIT_DIR / 'TEXTBOOK_INHERITANCE_EXAMPLES_v11.jsonl', textbook_audit_examples)
print(json.dumps(clear_audit, indent=2)[:12000])
if not clear_audit['clear_audit_passed']:
    raise RuntimeError('Clear audit failed. Promotion will not run.')

In [ ]:
# =========================
# Cell 11 — Write governed files and upload staged outputs
# =========================
# Governed outputs.
paths_out = {
    'textbook_chunks_governed': STAGE_DIR / 'textbook_primary_tagged_chunks_CURRICULUM_HARDENED_v11.jsonl',
    'textbook_pages_governed': STAGE_DIR / 'textbook_primary_tagged_pages_CURRICULUM_HARDENED_v11.jsonl',
    'textbook_vector_docstore_governed': STAGE_DIR / 'textbook_lean_vector_docstore_CURRICULUM_HARDENED_v11.jsonl',
    'pathout_docstore_governed': STAGE_DIR / 'pathout_ap_diagnostic_vector_docstore_CURRICULUM_HARDENED_v11.jsonl',
    'pathout_clean_pages_governed': STAGE_DIR / 'pathout_tagged_pages_AP_DIAGNOSTIC_CURRICULUM_HARDENED_v11.jsonl',
    'lecture_tag_map_governed': STAGE_DIR / 'lecture_primary_tag_map_STRICT_CYTO_CURRICULUM_HARDENED_v11.jsonl',
    'lecture_routed_chunks_governed': STAGE_DIR / 'lecture_timecoded_tagged_chunks_ROUTED_ONLY_STRICT_CYTO_CURRICULUM_HARDENED_v11.jsonl',
    'lecture_vector_docstore_governed': STAGE_DIR / 'lecture_timecoded_vector_docstore_STRICT_CYTO_CURRICULUM_HARDENED_v11.jsonl',
    'textbook_figure_map_governed': STAGE_DIR / 'textbook_figure_web_map_CURRICULUM_HARDENED_v11.jsonl',
}
write_jsonl(paths_out['textbook_chunks_governed'], textbook_chunks_gov)
if textbook_pages_gov: write_jsonl(paths_out['textbook_pages_governed'], textbook_pages_gov)
write_jsonl(paths_out['textbook_vector_docstore_governed'], textbook_vector_docstore_gov)
write_jsonl(paths_out['pathout_docstore_governed'], pathout_docstore_gov)
if pathout_clean_pages_gov: write_jsonl(paths_out['pathout_clean_pages_governed'], pathout_clean_pages_gov)
write_jsonl(paths_out['lecture_tag_map_governed'], lecture_tag_map_gov)
write_jsonl(paths_out['lecture_routed_chunks_governed'], lecture_routed_chunks_gov)
write_jsonl(paths_out['lecture_vector_docstore_governed'], lecture_docstore_gov)
if textbook_figure_map_gov: write_jsonl(paths_out['textbook_figure_map_governed'], textbook_figure_map_gov)

# Manifest summary patching.
def patch_manifest(manifest, source_name, rows, schema_version):
    m = dict(manifest or {})
    tags = [governed_tag_of(r) for r in rows]
    tag_counts = {}
    status_counts = {}
    for r, t in zip(rows, tags):
        tag_counts[t] = tag_counts.get(t, 0) + 1
        st = r.get('tag_governance_status', 'missing_status')
        status_counts[st] = status_counts.get(st, 0) + 1
    m.update({
        'schema_version': schema_version,
        'governance_version': 'curriculum_hardening_v11',
        'governance_generated_at_utc': now(),
        'record_count': len(rows),
        'mapped_count': sum(1 for t in tags if t != "__UNMAPPED__" and not is_junk_tag(t)),
        'unmapped_count': sum(1 for t in tags if t == "__UNMAPPED__"),
        'junk_visible_primary_tag_count': sum(1 for t in tags if t != "__UNMAPPED__" and is_junk_tag(t)),
        'unique_primary_tags': len(set(t for t in tags if t != "__UNMAPPED__" and not is_junk_tag(t))),
        'tag_governance_status_counts': status_counts,
        'top_primary_tags_governed': sorted(tag_counts.items(), key=lambda kv: kv[1], reverse=True)[:50],
        'notes': ['Curriculum hardening v11: ABPath gold + WHO fuzzy>=90 + PathOut auto-approved local + lecture/textbook sequence inheritance.'],
    })
    return m

textbook_manifest_gov = patch_manifest(textbook_manifest, 'textbooks', textbook_vector_docstore_gov, 'textbook_lean_vector_manifest.v11_curriculum_hardening_metadata_patch')
pathout_manifest_gov = patch_manifest(pathout_manifest, 'pathout', pathout_docstore_gov, 'pathout_vector_manifest.v11_curriculum_hardening_metadata_patch')
lecture_manifest_gov = patch_manifest(lecture_manifest, 'lectures', lecture_docstore_gov, 'lecture_timecoded_vector_manifest.STRICT_CYTO_v11_curriculum_hardening_metadata_patch')
manifest_paths = {
    'textbook_vector_manifest_governed': STAGE_DIR / 'textbook_lean_vector_manifest_CURRICULUM_HARDENED_v11.json',
    'pathout_vector_manifest_governed': STAGE_DIR / 'pathout_ap_diagnostic_vector_manifest_CURRICULUM_HARDENED_v11.json',
    'lecture_vector_manifest_governed': STAGE_DIR / 'lecture_timecoded_vector_manifest_STRICT_CYTO_CURRICULUM_HARDENED_v11.json',
}
write_json(manifest_paths['textbook_vector_manifest_governed'], textbook_manifest_gov)
write_json(manifest_paths['pathout_vector_manifest_governed'], pathout_manifest_gov)
write_json(manifest_paths['lecture_vector_manifest_governed'], lecture_manifest_gov)

# Copy index/browser/audits into stage.
sh(f"cp {index_path} {STAGE_DIR / index_path.name}")
for p in [OUT_DIR/'approved_curriculum_tag_tree_v11.json', OUT_DIR/'approved_curriculum_tag_catalog_v11.csv', OUT_DIR/'approved_curriculum_tag_browser_v11.html']:
    shutil.copy2(p, STAGE_DIR / p.name)

# Stage upload.
for p in STAGE_DIR.glob('*'):
    gcs_cp_to_uri(p, f"{GCS_STAGE_PREFIX}/{p.name}")
for p in AUDIT_DIR.glob('*'):
    gcs_cp_to_uri(p, f"{GCS_AUDIT_PREFIX}/{p.name}")
write_json(OUT_DIR / 'STAGED_OUTPUT_PATHS_v11.json', {
    'gcs_stage_prefix': GCS_STAGE_PREFIX,
    'gcs_audit_prefix': GCS_AUDIT_PREFIX,
    'files': {p.name: f"{GCS_STAGE_PREFIX}/{p.name}" for p in STAGE_DIR.glob('*')}
})
print("Staged outputs uploaded.")

In [ ]:
# =========================
# Cell 12 — Promotion, Cloud Run restart, and API proof
# =========================
# Target map: backend-consumed live paths replaced after backup.
promotion_map = {
    str(paths_out['textbook_chunks_governed']): GCS['textbook_chunks_live'],
    str(paths_out['textbook_vector_docstore_governed']): GCS['textbook_vector_docstore_live'],
    str(manifest_paths['textbook_vector_manifest_governed']): GCS['textbook_vector_manifest_live'],
    str(paths_out['pathout_docstore_governed']): GCS['pathout_vector_docstore_live'],
    str(manifest_paths['pathout_vector_manifest_governed']): GCS['pathout_vector_manifest_live'],
    str(paths_out['lecture_tag_map_governed']): GCS['lecture_tag_map_live'],
    str(paths_out['lecture_routed_chunks_governed']): GCS['lecture_routed_chunks_live'],
    str(paths_out['lecture_vector_docstore_governed']): GCS['lecture_vector_docstore_live'],
    str(manifest_paths['lecture_vector_manifest_governed']): GCS['lecture_vector_manifest_live'],
    str(STAGE_DIR / index_path.name): GCS_TAG_INDEX_LIVE,
}
if textbook_pages_gov:
    promotion_map[str(paths_out['textbook_pages_governed'])] = GCS['textbook_pages_live']
if pathout_clean_pages_gov:
    promotion_map[str(paths_out['pathout_clean_pages_governed'])] = GCS['pathout_clean_pages_live']
if CLEAN_TEXTBOOK_FIGURES and textbook_figure_map_gov and local_artifacts.get('textbook_figure_map_live'):
    promotion_map[str(paths_out['textbook_figure_map_governed'])] = GCS['textbook_figure_map_live']

promotion_audit = {'schema_version': 'pathology_hub_curriculum_tag_hardening_promotion.v11', 'generated_at_utc': now(), 'promotion_mode': PROMOTION_MODE, 'performed': False, 'targets': [], 'cloud_run_restart_requested': False}

if PROMOTION_MODE == "backup_replace_live":
    if not clear_audit.get('clear_audit_passed'):
        raise RuntimeError('Promotion blocked: clear_audit_passed is false.')
    for src_local, dst_uri in promotion_map.items():
        backup_uri = f"{GCS_BACKUP_PREFIX}/{dst_uri.replace('gs://','').replace('/','__')}"
        if gcs_exists(dst_uri):
            sh(f"gcloud storage cp {dst_uri} {backup_uri}", check=True, timeout=1200)
        gcs_cp_to_uri(src_local, dst_uri, check=True, timeout=1800)
        promotion_audit['targets'].append({'local_source': src_local, 'live_target': dst_uri, 'backup_uri': backup_uri})
    promotion_audit['performed'] = True
    if RESTART_CLOUD_RUN_AFTER_PROMOTION:
        p = sh(f"gcloud run services update {LIVE_SERVICE} --region {REGION} --project {PROJECT_ID} --update-env-vars CURRICULUM_TAG_GOVERNANCE_VERSION=v11_{RUN_TS}", check=False, timeout=900)
        promotion_audit['cloud_run_restart_requested'] = True
        promotion_audit['cloud_run_restart_returncode'] = p.returncode
        promotion_audit['cloud_run_restart_output_tail'] = p.stdout[-4000:] if p.stdout else ''
elif PROMOTION_MODE == "none":
    promotion_audit['notes'] = ['Promotion disabled; staged outputs only.']
else:
    raise ValueError(f"Unknown PROMOTION_MODE: {PROMOTION_MODE}")

write_json(AUDIT_DIR / 'CURRICULUM_TAG_HARDENING_V11_PROMOTION_AUDIT.json', promotion_audit)
gcs_cp_to_uri(AUDIT_DIR / 'CURRICULUM_TAG_HARDENING_V11_PROMOTION_AUDIT.json', f"{GCS_AUDIT_PREFIX}/CURRICULUM_TAG_HARDENING_V11_PROMOTION_AUDIT.json")
print(json.dumps(promotion_audit, indent=2)[:8000])

# Health and API proof.
proof = {'schema_version': 'pathology_hub_curriculum_tag_hardening_api_proof.v11', 'generated_at_utc': now(), 'health': None, 'search_probes': [], 'api_key_available': bool(API_KEY)}
try:
    hr = requests.get(HEALTH_URL, timeout=60)
    proof['health'] = {'status_code': hr.status_code, 'text_excerpt': hr.text[:4000]}
    try: proof['health_json'] = hr.json()
    except Exception: pass
except Exception as e:
    proof['health_error'] = str(e)

if RUN_API_PROOF:
    headers = {'Content-Type': 'application/json'}
    if API_KEY:
        headers['X-API-Key'] = API_KEY
    for source, query in [('textbooks','high grade serous carcinoma p53'), ('lectures','melanoma overview'), ('pathout','ovary clear cell carcinoma'), ('who','serous carcinoma ovary')]:
        payload = {'query': query, 'sources': [source], 'max_results': 3, 'include_figures': False, 'max_figures': 0, 'compact': True, 'excerpt_char_limit': 800}
        item = {'source': source, 'query': query}
        try:
            rr = requests.post(SEARCH_URL, headers=headers, json=payload, timeout=90)
            item['status_code'] = rr.status_code
            item['text_excerpt'] = rr.text[:3000]
            if rr.status_code == 200:
                data = rr.json()
                item['json_keys'] = list(data.keys())[:20]
                tags = []
                for key, val in data.items():
                    if key.endswith('_results') and isinstance(val, list):
                        for rec in val:
                            t = rec.get('primary_tag') or rec.get('primary_tag_governed')
                            if t: tags.append(t)
                item['returned_primary_tags'] = tags
                item['junk_returned_primary_tags'] = [t for t in tags if is_junk_tag(t)]
        except Exception as e:
            item['error'] = str(e)
        proof['search_probes'].append(item)
proof['api_proof_passed'] = all(not p.get('junk_returned_primary_tags') for p in proof.get('search_probes', []) if p.get('status_code') == 200)
write_json(AUDIT_DIR / 'CURRICULUM_TAG_HARDENING_V11_API_PROOF.json', proof)
gcs_cp_to_uri(AUDIT_DIR / 'CURRICULUM_TAG_HARDENING_V11_API_PROOF.json', f"{GCS_AUDIT_PREFIX}/CURRICULUM_TAG_HARDENING_V11_API_PROOF.json")
print(json.dumps(proof, indent=2)[:12000])

In [ ]:
# =========================
# Cell 13 — Final report and downloadable ZIP
# =========================
report = {
    'schema_version': 'pathology_hub_curriculum_tag_hardening_run_report.v11',
    'generated_at_utc': now(),
    'workstream': 'Evidence/Lesson/Research RAG / curriculum mapping tag governance',
    'purpose': 'Harden curriculum tags with ABPath/WHO/PathOut-approved tags, sequence inheritance, and junk-tag/figure suppression.',
    'clear_audit': clear_audit,
    'promotion': promotion_audit,
    'api_proof_summary': {
        'api_key_available': proof.get('api_key_available'),
        'api_proof_passed': proof.get('api_proof_passed'),
        'health_status_code': (proof.get('health') or {}).get('status_code'),
    },
    'gcs_stage_prefix': GCS_STAGE_PREFIX,
    'gcs_audit_prefix': GCS_AUDIT_PREFIX,
    'gcs_backup_prefix': GCS_BACKUP_PREFIX,
    'gcs_tag_index_live': GCS_TAG_INDEX_LIVE,
    'known_limitations': [
        'No secondary morphology/IHC/molecular curriculum facets generated in this pass by user decision.',
        'WHO fuzzy matches below threshold are excluded pending review.',
        'Chunks with no approved mapping and no prior same-sequence context remain governed __UNMAPPED__ and are excluded from tag browsing.',
        'Raw PDFs/videos are preserved; only retrieval-facing metadata and derived figure map are patched.',
    ]
}
write_json(OUT_DIR / 'PATHOLOGY_HUB_CURRICULUM_TAG_HARDENING_V11_RUN_REPORT.json', report)

# Create markdown handoff.
md = f"""# Handoff — Curriculum Tag Hardening v11

Generated: {now()}

## Workstream
Evidence/Lesson/Research RAG — curriculum mapping tag governance.

## Purpose
Use ABPath gold tags, accepted WHO→ABPath matches, and approved PathOut local tags as the curriculum ontology. Suppress generated lecture/textbook junk tags by mapping or inheriting nearest meaningful context within sequence limits.

## Decisions implemented
- PathOut-only tags auto-approved as local curriculum tags.
- WHO fuzzy matches to ABPath auto-accepted at score >= {WHO_FUZZY_ACCEPT_THRESHOLD}.
- Lecture weak chunks inherit within {MAX_LECTURE_INHERIT_GAP_SEC} seconds / {MAX_LECTURE_INHERIT_ROW_GAP} rows.
- Textbook weak chunks inherit within {MAX_TEXTBOOK_INHERIT_PAGE_GAP} pages / {MAX_TEXTBOOK_INHERIT_ROW_GAP} rows.
- Chunks with no approved tag and no prior context remain governed `__UNMAPPED__` and are excluded from approved tag browsing.
- Secondary curriculum facets are intentionally held off.

## Promotion
Promotion mode: `{PROMOTION_MODE}`. Performed: `{promotion_audit.get('performed')}`.

## Outputs
- Governed textbook/pathout/lecture metadata JSONL.
- Governed vector manifests.
- Approved-only SQLite tag index.
- Approved curriculum tag browser/catalog/tree.
- WHO fuzzy-match audit.
- Figure exclusion audit.
- Promotion audit and API proof.

## GCS
Stage: `{GCS_STAGE_PREFIX}`  
Audits: `{GCS_AUDIT_PREFIX}`  
Backups: `{GCS_BACKUP_PREFIX}`  
Live approved tag index: `{GCS_TAG_INDEX_LIVE}`

## Caveats
Raw sources are preserved. No embeddings or FAISS indexes are rebuilt. `__UNMAPPED__` governed rows may remain, but they are not visible in approved tag browsing.
"""
(OUT_DIR / 'HANDOFF_CURRICULUM_TAG_HARDENING_V11.md').write_text(md, encoding='utf-8')

# Checksums.
manifest = {}
for p in OUT_DIR.rglob('*'):
    if p.is_file():
        manifest[str(p.relative_to(OUT_DIR))] = {'sha256': sha256_file(p), 'size_bytes': p.stat().st_size}
write_json(OUT_DIR / 'SHA256SUMS_v11.json', manifest)

zip_path = WORKDIR / f"PATHOLOGY_HUB_CURRICULUM_TAG_HARDENING_V11_OUTPUTS_{RUN_TS}.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob('*'):
        if p.is_file():
            z.write(p, p.relative_to(OUT_DIR))
print("Output ZIP:", zip_path)
files.download(str(zip_path))